In [1]:
import os
import sys
import shutil
from contextlib import contextmanager

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

SRC_ROOT = os.path.abspath('src')
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import util.feds_util as feds_util
import util.general_util as gen_util
import util.processing_util as proc_util


In [2]:
# ---- Config ----
# EVENT_ID = 'CA3858612053820210815'
# EVENT_ID = 'CA3442911910020171205'
EVENT_ID = 'CA4145912232920210625'
# EVENT_ID = "ID4610011544620150814"
DATA_ROOT = '/extra/datalab_scratch0/firecube_data_new/inputData'
CUBES_ROOT = '/extra/datalab_scratch0/firecube_data_new/output/cubes'
FEDS_ROOT = os.path.join(DATA_ROOT, 'Full_FEDS')
FIRELIST_CSV = os.path.join(DATA_ROOT, 'fireslist2012-2024_withtype_no_fline_hawaii.csv')
FIREPIX_ROOT = os.path.join(DATA_ROOT, 'firepix')
OUT_ROOT = os.path.join('temp', 'feds_ab_test_methods', EVENT_ID)
RES_M = 300.0
USE_PREV = False
DO_PLOTS = False

# Point util paths to production-style data roots
feds_util.dir_feds25 = FEDS_ROOT
feds_util.feds_firelist = FIRELIST_CSV
feds_util.dir_firepix = FIREPIX_ROOT

print('EVENT_ID:', EVENT_ID)
print('CUBES_ROOT:', CUBES_ROOT)
print('FEDS_ROOT:', FEDS_ROOT)
print('FIRELIST_CSV:', FIRELIST_CSV)
print('FIREPIX_ROOT:', FIREPIX_ROOT)
print('OUT_ROOT:', OUT_ROOT)


EVENT_ID: CA4145912232920210625
CUBES_ROOT: /extra/datalab_scratch0/firecube_data_new/output/cubes
FEDS_ROOT: /extra/datalab_scratch0/firecube_data_new/inputData/Full_FEDS
FIRELIST_CSV: /extra/datalab_scratch0/firecube_data_new/inputData/fireslist2012-2024_withtype_no_fline_hawaii.csv
FIREPIX_ROOT: /extra/datalab_scratch0/firecube_data_new/inputData/firepix
OUT_ROOT: temp/feds_ab_test_methods/CA4145912232920210625


In [3]:
def _first_existing_non_feds_tif(event_dir):
    candidate_batches = ['low_res_climate', 'high_res_climate', 'fuel_topo', 'landfire']
    for batch in candidate_batches:
        bdir = os.path.join(event_dir, batch)
        if not os.path.isdir(bdir):
            continue
        for fn in sorted(os.listdir(bdir)):
            if fn.endswith('.tif'):
                return os.path.join(bdir, fn)
    raise FileNotFoundError(f'No non-FEDS tif found under {event_dir}')


def _compute_main_time_params(event_id, gdf_fperim_rd):
    firelist = pd.read_csv(feds_util.feds_firelist)
    row = firelist[firelist['Event_ID'] == event_id]
    if row.empty:
        raise RuntimeError(f'{event_id} missing in {feds_util.feds_firelist}')

    center_lon = round((float(row['lon0'].iloc[0]) + float(row['lon1'].iloc[0])) / 2.0, 2)
    conversion_delta = pd.to_timedelta(1, unit='hours') - pd.to_timedelta(round(center_lon / 15), unit='hours')

    df_t = pd.to_datetime(gdf_fperim_rd.t)
    df_t_with_buffer = proc_util.add_time_buffers(df_t) + conversion_delta

    fire_start = pd.Timestamp(df_t_with_buffer.min().normalize())
    fire_end = pd.Timestamp(df_t_with_buffer.max().normalize()) + pd.Timedelta(hours=23)
    fire_hours = int((fire_end - fire_start).total_seconds() / 3600)
    return conversion_delta, fire_start, fire_end, fire_hours


def _safe_rm_tree(path):
    if os.path.isdir(path):
        shutil.rmtree(path)


def _raster_meta(path):
    with rasterio.open(path) as src:
        return {
            'count': src.count,
            'height': src.height,
            'width': src.width,
            'res': src.res,
            'bounds': (src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top),
            'crs': str(src.crs),
            'transform': tuple(src.transform),
        }


@contextmanager
def _override_gen_paths(temp_root, output_root):
    old_temp = gen_util.dir_temp
    old_output = gen_util.dir_output
    gen_util.dir_temp = temp_root
    gen_util.dir_output = output_root
    try:
        yield
    finally:
        gen_util.dir_temp = old_temp
        gen_util.dir_output = old_output


def _run_pipeline_with_driver_methods(
    event_id,
    final_bounds,
    fire_start,
    fire_hours,
    conversion_delta,
    run_root,
    direct_to_final_grid,
):
    temp_root = os.path.join(run_root, 'temp')
    output_root = os.path.join(run_root, 'output')

    _safe_rm_tree(temp_root)
    _safe_rm_tree(output_root)
    os.makedirs(run_root, exist_ok=True)

    with _override_gen_paths(temp_root=temp_root, output_root=output_root):
        gen_util.create_dirs_for_fire(event_id)

        # Call the methods under test directly.
        feds_util.driver_feds(
            event_id,
            final_bounds,
            res=RES_M,
            fire_start=fire_start,
            num_hours=fire_hours,
            plot_orig=DO_PLOTS,
            use_prev=USE_PREV,
            conv_delta=conversion_delta,
            direct_to_final_grid=direct_to_final_grid,
        )
        feds_util.driver_frp(
            event_id,
            final_bounds,
            res=RES_M,
            fire_start=fire_start,
            num_hours=fire_hours,
            plot_orig=DO_PLOTS,
            use_prev=USE_PREV,
            conv_delta=conversion_delta,
            direct_to_final_grid=direct_to_final_grid,
        )

        fire_spread_dir = os.path.join(gen_util.dir_output, gen_util.dir_cubes, event_id, 'fire_spread')
        fire_times_csv = os.path.join(gen_util.dir_output, gen_util.dir_cubes, event_id, 'fire_times.csv')

    return fire_spread_dir, fire_times_csv, temp_root, output_root


In [4]:
event_dir = os.path.join(CUBES_ROOT, EVENT_ID)
if not os.path.isdir(event_dir):
    raise RuntimeError(f'Missing event dir: {event_dir}')

non_feds_ref_tif = _first_existing_non_feds_tif(event_dir)
final_bounds = proc_util.get_tif_bounds(non_feds_ref_tif)
final_transform, final_width, final_height = feds_util.get_canonical_grid_from_bounds(final_bounds, RES_M)

gdf_fperim_rd, gdf_fline_rd, gdf_nfp_rd = feds_util.read_1fire(EVENT_ID)
if gdf_fperim_rd is None or gdf_fperim_rd.empty:
    raise RuntimeError(f'No perimeter GPKG data for {EVENT_ID}')
if gdf_nfp_rd is None or gdf_nfp_rd.empty:
    raise RuntimeError(f'No newfirepix GPKG data for {EVENT_ID}')

conversion_delta, fire_start, fire_end, fire_hours = _compute_main_time_params(EVENT_ID, gdf_fperim_rd)
df_fp = feds_util.read_firepix_1fire(gdf_fperim_rd.t.min().year, EVENT_ID)

existing_fperim = os.path.join(event_dir, 'fire_spread', 'fperim.tif')
existing_count = None
if os.path.exists(existing_fperim):
    with rasterio.open(existing_fperim) as src:
        existing_count = src.count

print('Using non-FEDS reference tif:', non_feds_ref_tif)
print('Final bounds:', final_bounds)
print('Final grid (w, h):', final_width, final_height)
print('Conversion delta (LST -> UTC):', conversion_delta)
print('Fire window (main.py style):', fire_start, '->', fire_end, '| hours:', fire_hours)
print('Existing fperim band count:', existing_count)
print('FEDS rows:', len(gdf_fperim_rd), '| fline rows:', len(gdf_fline_rd), '| nfp rows:', len(gdf_nfp_rd), '| firepix rows:', len(df_fp))


Using non-FEDS reference tif: /extra/datalab_scratch0/firecube_data_new/output/cubes/CA4145912232920210625/low_res_climate/d2m.tif
Final bounds: [-2159044.19355312  2340257.08476283 -2141044.19355312  2367257.08476283]
Final grid (w, h): 60 90
Conversion delta (LST -> UTC): 0 days 09:00:00
Fire window (main.py style): 2021-06-25 00:00:00 -> 2021-07-20 23:00:00 | hours: 623
Existing fperim band count: 48
FEDS rows: 48 | fline rows: 33 | nfp rows: 48 | firepix rows: 1493


In [5]:
legacy_run_root = os.path.join(OUT_ROOT, 'legacy_run')
direct_run_root = os.path.join(OUT_ROOT, 'direct_run')

legacy_dir, legacy_times_csv, legacy_temp_root, legacy_output_root = _run_pipeline_with_driver_methods(
    event_id=EVENT_ID,
    final_bounds=final_bounds,
    fire_start=fire_start,
    fire_hours=fire_hours,
    conversion_delta=conversion_delta,
    run_root=legacy_run_root,
    direct_to_final_grid=False,
)

direct_dir, direct_times_csv, direct_temp_root, direct_output_root = _run_pipeline_with_driver_methods(
    event_id=EVENT_ID,
    final_bounds=final_bounds,
    fire_start=fire_start,
    fire_hours=fire_hours,
    conversion_delta=conversion_delta,
    run_root=direct_run_root,
    direct_to_final_grid=True,
)

print('Legacy fire_spread dir:', legacy_dir)
print('Direct fire_spread dir:', direct_dir)
print('Legacy fire_times.csv:', legacy_times_csv, '| exists:', os.path.exists(legacy_times_csv))
print('Direct fire_times.csv:', direct_times_csv, '| exists:', os.path.exists(direct_times_csv))

for var in ['fperim', 'fline', 'nfp', 'frp']:
    p_leg = os.path.join(legacy_dir, f'{var}.tif')
    p_dir = os.path.join(direct_dir, f'{var}.tif')
    print(f'{var:6s} legacy exists={os.path.exists(p_leg)} direct exists={os.path.exists(p_dir)}')


/home/gmiglior/.local/lib/python3.10/site-packages/numpy/_core/_asarray.py:127: RuntimeWarning: invalid value encountered in cast
  arr = array(a, dtype=dtype, order=order, copy=None, subok=subok)


Legacy fire_spread dir: temp/feds_ab_test_methods/CA4145912232920210625/legacy_run/output/cubes/CA4145912232920210625/fire_spread
Direct fire_spread dir: temp/feds_ab_test_methods/CA4145912232920210625/direct_run/output/cubes/CA4145912232920210625/fire_spread
Legacy fire_times.csv: temp/feds_ab_test_methods/CA4145912232920210625/legacy_run/output/cubes/CA4145912232920210625/fire_times.csv | exists: True
Direct fire_times.csv: temp/feds_ab_test_methods/CA4145912232920210625/direct_run/output/cubes/CA4145912232920210625/fire_times.csv | exists: True
fperim legacy exists=True direct exists=True
fline  legacy exists=True direct exists=True
nfp    legacy exists=True direct exists=True
frp    legacy exists=True direct exists=True


In [6]:
def _read_presence_stack(path):
    with rasterio.open(path) as src:
        arr = np.nan_to_num(src.read().astype(np.float32), nan=0.0)
    return arr > 0

def _mean_iou(path_a, path_b, max_bands=72):
    a = _read_presence_stack(path_a)
    b = _read_presence_stack(path_b)
    n = min(a.shape[0], b.shape[0], max_bands)
    vals = []
    for i in range(n):
        inter = np.logical_and(a[i], b[i]).sum()
        union = np.logical_or(a[i], b[i]).sum()
        vals.append(float(inter / union) if union > 0 else np.nan)
    return float(np.nanmean(vals)) if len(vals) > 0 else np.nan, n

vars_all = ['fperim', 'fline', 'nfp', 'frp']
existing_dir = os.path.join(event_dir, 'fire_spread')

rows = []
for var in vars_all:
    p_legacy = os.path.join(legacy_dir, f'{var}.tif')
    p_direct = os.path.join(direct_dir, f'{var}.tif')
    p_exist = os.path.join(existing_dir, f'{var}.tif')

    if not (os.path.exists(p_legacy) and os.path.exists(p_direct)):
        continue

    m_legacy = _raster_meta(p_legacy)
    m_direct = _raster_meta(p_direct)
    iou_ld, n_ld = _mean_iou(p_legacy, p_direct)

    row = {
        'var': var,
        'legacy_vs_direct_iou': iou_ld,
        'legacy_vs_direct_bands': n_ld,
        'legacy_shape': (m_legacy['count'], m_legacy['height'], m_legacy['width']),
        'direct_shape': (m_direct['count'], m_direct['height'], m_direct['width']),
        'legacy_res': m_legacy['res'],
        'direct_res': m_direct['res'],
    }

    if os.path.exists(p_exist):
        iou_el, n_el = _mean_iou(p_exist, p_legacy)
        iou_ed, n_ed = _mean_iou(p_exist, p_direct)
        row['existing_vs_legacy_iou'] = iou_el
        row['existing_vs_legacy_bands'] = n_el
        row['existing_vs_direct_iou'] = iou_ed
        row['existing_vs_direct_bands'] = n_ed

    rows.append(row)

cmp_df = pd.DataFrame(rows)
display(cmp_df)


,var,legacy_vs_direct_iou,legacy_vs_direct_bands,legacy_shape,direct_shape,legacy_res,direct_res,existing_vs_legacy_iou,existing_vs_legacy_bands,existing_vs_direct_iou,existing_vs_direct_bands
0,fperim,0.856197,48,"(48, 90, 60)","(48, 90, 60)","(300.0, 300.0)","(300.0, 300.0)",1.0,48,0.856197,48
1,fline,0.208552,48,"(48, 90, 60)","(48, 90, 60)","(300.0, 300.0)","(300.0, 300.0)",1.0,48,0.208552,48
2,nfp,0.240303,48,"(48, 90, 60)","(48, 90, 60)","(300.0, 300.0)","(300.0, 300.0)",1.0,48,0.240303,48
3,frp,0.298351,48,"(48, 90, 60)","(48, 90, 60)","(300.0, 300.0)","(300.0, 300.0)",1.0,48,0.298351,48


In [8]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from rasterio.warp import reproject, Resampling

def _read_bool_stack(path):
    with rasterio.open(path) as src:
        data = np.nan_to_num(src.read().astype(np.float32), nan=0.0) > 0
        return data, src.transform, src.crs

def _reproject_mask(mask, src_transform, src_crs, dst_shape, dst_transform, dst_crs):
    out = np.zeros(dst_shape, dtype=np.uint8)
    reproject(
        source=mask.astype(np.uint8),
        destination=out,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        src_nodata=0,
        dst_nodata=0,
        resampling=Resampling.nearest,
    )
    return out > 0

def _mask_bounds(mask, transform):
    if mask is None or not np.any(mask):
        return None
    rr, cc = np.where(mask)
    r0, r1 = int(rr.min()), int(rr.max()) + 1
    c0, c1 = int(cc.min()), int(cc.max()) + 1
    left, top = transform * (c0, r0)
    right, bottom = transform * (c1, r1)
    return [min(left, right), min(bottom, top), max(left, right), max(bottom, top)]

def _union_bounds(bounds_list):
    vals = [b for b in bounds_list if b is not None]
    if not vals:
        return None
    return [
        min(v[0] for v in vals),
        min(v[1] for v in vals),
        max(v[2] for v in vals),
        max(v[3] for v in vals),
    ]

def _add_buffer(bounds, meters):
    return [bounds[0]-meters, bounds[1]-meters, bounds[2]+meters, bounds[3]+meters]

def _rgba(mask, rgb, alpha=0.5):
    arr = np.zeros((mask.shape[0], mask.shape[1], 4), dtype=np.float32)
    arr[mask, 0] = rgb[0]
    arr[mask, 1] = rgb[1]
    arr[mask, 2] = rgb[2]
    arr[mask, 3] = alpha
    return arr

def _rasterize_gdf_at_time(gdf, t_local, out_shape, transform):
    g_t = gdf[gdf['t'] == pd.Timestamp(t_local)].dropna(subset=['geometry'])
    if g_t.empty:
        return np.zeros(out_shape, dtype=bool)
    ras = rasterio.features.rasterize(
        ((geom, 1) for geom in g_t.geometry if geom is not None and (not geom.is_empty)),
        out_shape=out_shape,
        transform=transform,
        fill=0,
        all_touched=True,
        dtype='uint8',
    )
    return ras > 0

def _iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union > 0 else np.nan

def _plot_ref_geom(ax, gdf_t, color='#ffd54f', lw=1.2, ls='-.'):
    if gdf_t is None or gdf_t.empty:
        return
    try:
        geom_types = set(gdf_t.geometry.geom_type.dropna().tolist())
        if any('Polygon' in gt for gt in geom_types):
            gdf_t.boundary.plot(ax=ax, color=color, linewidth=lw, linestyle=ls)
        else:
            gdf_t.plot(ax=ax, color=color, linewidth=lw)
    except Exception:
        pass

def _make_hillshade(event_dir, fallback_extent):
    dem_candidates = [
        os.path.join(event_dir, 'fuel_topo', 'dem.tif'),
        os.path.join(event_dir, 'landfire', 'elev2020.tif'),
        os.path.join(event_dir, 'landfire', 'ak_elev2020.tif'),
        os.path.join(event_dir, 'landfire', 'hi_elev2020.tif'),
    ]
    dem_path = next((p for p in dem_candidates if os.path.exists(p)), None)

    if dem_path is None:
        return None, fallback_extent

    with rasterio.open(dem_path) as ds_dem:
        dem = ds_dem.read(1).astype(np.float32)
        transform = ds_dem.transform
        extent = [ds_dem.bounds.left, ds_dem.bounds.right, ds_dem.bounds.bottom, ds_dem.bounds.top]

    if np.isfinite(dem).any():
        fill_val = float(np.nanmedian(dem[np.isfinite(dem)]))
        dem_fill = np.nan_to_num(dem, nan=fill_val)
    else:
        dem_fill = np.zeros_like(dem, dtype=np.float32)

    sx = abs(float(transform.a))
    sy = abs(float(transform.e))
    dy, dx = np.gradient(dem_fill, sy, sx)
    slope = np.pi / 2.0 - np.arctan(np.hypot(dx, dy))
    aspect = np.arctan2(-dx, dy)
    az = np.deg2rad(315.0)
    alt = np.deg2rad(45.0)
    hs = np.sin(alt) * np.sin(slope) + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
    hs = np.clip(hs, 0.0, 1.0)
    return hs, extent

existing_dir = os.path.join(event_dir, 'fire_spread')
legacy_paths = {v: os.path.join(legacy_dir, f'{v}.tif') for v in ['fperim', 'fline', 'nfp']}
direct_paths = {v: os.path.join(direct_dir, f'{v}.tif') for v in ['fperim', 'fline', 'nfp']}
existing_paths = {v: os.path.join(existing_dir, f'{v}.tif') for v in ['fperim', 'fline', 'nfp']}

for v in ['fperim', 'fline', 'nfp']:
    if not (os.path.exists(legacy_paths[v]) and os.path.exists(direct_paths[v]) and os.path.exists(existing_paths[v])):
        raise FileNotFoundError(f'Missing tif for {v}: legacy/direct/existing required')

legacy_stacks = {}
direct_stacks = {}
existing_stacks = {}

# Use legacy grid as base plotting grid
for v in ['fperim', 'fline', 'nfp']:
    legacy_stacks[v], base_transform, base_crs = _read_bool_stack(legacy_paths[v])

base_shape = legacy_stacks['fperim'].shape[1:]

# Reproject direct/existing stacks to base grid if needed
for v in ['fperim', 'fline', 'nfp']:
    direct_data, direct_transform, direct_crs = _read_bool_stack(direct_paths[v])
    existing_data, existing_transform, existing_crs = _read_bool_stack(existing_paths[v])

    direct_out = np.zeros((direct_data.shape[0], base_shape[0], base_shape[1]), dtype=bool)
    for i in range(direct_data.shape[0]):
        direct_out[i] = _reproject_mask(direct_data[i], direct_transform, direct_crs, base_shape, base_transform, base_crs)

    existing_out = np.zeros((existing_data.shape[0], base_shape[0], base_shape[1]), dtype=bool)
    for i in range(existing_data.shape[0]):
        existing_out[i] = _reproject_mask(existing_data[i], existing_transform, existing_crs, base_shape, base_transform, base_crs)

    direct_stacks[v] = direct_out
    existing_stacks[v] = existing_out

left, top = base_transform * (0, 0)
right, bottom = base_transform * (base_shape[1], base_shape[0])
base_extent = [min(left, right), max(left, right), min(bottom, top), max(bottom, top)]

# Prepare vector refs in base CRS
gdf_perim = gdf_fperim_rd.to_crs(base_crs)
gdf_fline = gdf_fline_rd.to_crs(base_crs)
gdf_nfp = gdf_nfp_rd.to_crs(base_crs)
layer_to_gdf = {'fperim': gdf_perim, 'fline': gdf_fline, 'nfp': gdf_nfp}

perim_times = sorted(pd.to_datetime(gdf_perim['t']).dropna().unique())
n_frames = min(len(perim_times), legacy_stacks['fperim'].shape[0], direct_stacks['fperim'].shape[0], existing_stacks['fperim'].shape[0])
if n_frames == 0:
    raise RuntimeError('No frames available for plotting')

plot_idx = [i for i in range(n_frames) if legacy_stacks['fperim'][i].any() or direct_stacks['fperim'][i].any() or existing_stacks['fperim'][i].any()]
if len(plot_idx) == 0:
    plot_idx = list(range(n_frames))
plot_idx = plot_idx[:4]

hs_native, bg_extent = _make_hillshade(event_dir, base_extent)

fig, axes = plt.subplots(nrows=len(plot_idx), ncols=3, figsize=(18, 5 * len(plot_idx)), dpi=190, squeeze=False)

for row_i, obs_i in enumerate(plot_idx):
    t_local = pd.Timestamp(perim_times[obs_i])

    # Zoom window based on perimeter union
    perim_ref_mask = _rasterize_gdf_at_time(gdf_perim, t_local, base_shape, base_transform)
    b_legacy = _mask_bounds(legacy_stacks['fperim'][obs_i], base_transform)
    b_direct = _mask_bounds(direct_stacks['fperim'][obs_i], base_transform)
    b_existing = _mask_bounds(existing_stacks['fperim'][obs_i], base_transform)
    b_ref = _mask_bounds(perim_ref_mask, base_transform)
    zb = _union_bounds([b_legacy, b_direct, b_existing, b_ref])
    if zb is None:
        zoom_extent = [base_extent[0], base_extent[1], base_extent[2], base_extent[3]]
    else:
        zbb = _add_buffer(zb, meters=RES_M * 6.0)
        zoom_extent = [zbb[0], zbb[2], zbb[1], zbb[3]]

    for col_i, var in enumerate(['fperim', 'fline', 'nfp']):
        ax = axes[row_i, col_i]

        legacy_mask = legacy_stacks[var][min(obs_i, legacy_stacks[var].shape[0]-1)]
        direct_mask = direct_stacks[var][min(obs_i, direct_stacks[var].shape[0]-1)]
        existing_mask = existing_stacks[var][min(obs_i, existing_stacks[var].shape[0]-1)]

        gdf_t = layer_to_gdf[var][layer_to_gdf[var]['t'] == t_local].dropna(subset=['geometry'])
        ref_mask = _rasterize_gdf_at_time(layer_to_gdf[var], t_local, base_shape, base_transform)

        if hs_native is not None:
            ax.imshow(hs_native, cmap='gray', extent=bg_extent, origin='upper', interpolation='nearest')

        ax.imshow(_rgba(existing_mask, rgb=(1.00, 0.18, 0.18), alpha=0.35), extent=base_extent, origin='upper', interpolation='nearest')
        # ax.imshow(_rgba(legacy_mask, rgb=(1.00, 0.18, 0.18), alpha=0.50), extent=base_extent, origin='upper', interpolation='nearest')
        ax.imshow(_rgba(direct_mask, rgb=(0.00, 0.74, 0.83), alpha=0.40), extent=base_extent, origin='upper', interpolation='nearest')
        _plot_ref_geom(ax, gdf_t, color='#ffd54f', lw=1.0, ls='-.')

        iou_ex = _iou(existing_mask, ref_mask)
        # iou_leg = _iou(legacy_mask, ref_mask)
        iou_dir = _iou(direct_mask, ref_mask)
        # ax.set_title(f'{EVENT_ID} | obs_i={obs_i} | {var}\nIoU ex={iou_ex:.3f}, leg={iou_leg:.3f}, dir={iou_dir:.3f}')
        ax.set_xlim(zoom_extent[0], zoom_extent[1])
        ax.set_ylim(zoom_extent[2], zoom_extent[3])
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')

handles = [
    Patch(facecolor=(0.20, 0.95, 0.25, 0.35), edgecolor='none', label='Existing TIFF'),
    Patch(facecolor=(1.00, 0.18, 0.18, 0.50), edgecolor='none', label='Legacy TIFF'),
    Patch(facecolor=(0.00, 0.74, 0.83, 0.40), edgecolor='none', label='Direct TIFF'),
    Line2D([0], [0], color='#ffd54f', lw=1.8, linestyle='-.', label='GPKG vector'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=True, bbox_to_anchor=(0.5, 0.01))
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()
# plt.savefig(f"problem.png")